#  Web Scraping с Selenium


## Базовая работа с браузером

In [38]:
# pip install selenium
# pip install webdriver-manager
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import time
import pandas as pd

In [40]:
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
driver.get("https://weather.com/science/environment/news/2025-10-15-surrey-smart-robot-fish-eleanor-mackintosh")

print("Заголовок:", driver.title)

print("URL:", driver.current_url)

driver.quit()

Заголовок: This Robotic Fish Feeds On Plastic Waste | Weather.com
URL: https://weather.com/science/environment/news/2025-10-15-surrey-smart-robot-fish-eleanor-mackintosh


##  Поиск элементов на странице

### Основные методы поиска:

- `By.ID` - поиск по id
- `By.CLASS_NAME` - поиск по классу
- `By.TAG_NAME` - поиск по тегу
- `By.CSS_SELECTOR` - CSS селектор
- `By.XPATH` - XPath выражение

In [41]:
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
driver.get("https://weather.com/science/environment/news/2025-10-15-surrey-smart-robot-fish-eleanor-mackintosh")

heading = driver.find_element(By.TAG_NAME, "h1")
print("Заголовок H1:", heading.text)

paragraphs = driver.find_elements(By.TAG_NAME, "p")
print(f"\nНайдено параграфов: {len(paragraphs)}")
for i, p in enumerate(paragraphs):
    print(f"Параграф {i+1}: {p.text[:100]}...")  

driver.quit()

Заголовок H1: This Plastic-Eating Robot Fish Could Help Clean The Planet

Найдено параграфов: 21
Параграф 1: An invention born from a contest at England's University of Surrey might be swimming us closer to cl...
Параграф 2: This “robo-fish,” designed to fight the growing crisis of microplastic contamination, uses a microbi...
Параграф 3: In other words, the more plastic it eats, the more energy it has to keep swimming, making it one of ...
Параграф 4: (MORE: Microplastics Could Change Our Weather)...
Параграф 5: The project began as part of the Natural Robots Contest at the University of Surrey, which invited p...
Параграф 6: From Mackintosh’s concept, university engineers brought Gillbert to life, transforming it into a foo...
Параграф 7: As it glides through oceans, rivers or lakes, the robo-fish keeps its mouth open to pull in water. I...
Параграф 8: Small onboard sensors track light levels and water quality, offering valuable environmental data alo...
Параграф 9: And get this: The

## Извлечение данных из таблицы

Практический пример: извлекаем данные из HTML таблицы

In [42]:
html_content = """
<!DOCTYPE html>
<html>
<head><title>Пример таблицы</title></head>
<body>
    <h1>Продажи продуктов</h1>
    <table id="sales-table" border="1">
        <thead>
            <tr>
                <th>Продукт</th>
                <th>Цена</th>
                <th>Количество</th>
            </tr>
        </thead>
        <tbody>
            <tr>
                <td>Ноутбук</td>
                <td>50000</td>
                <td>15</td>
            </tr>
            <tr>
                <td>Мышь</td>
                <td>500</td>
                <td>120</td>
            </tr>
            <tr>
                <td>Клавиатура</td>
                <td>1500</td>
                <td>80</td>
            </tr>
            <tr>
                <td>Монитор</td>
                <td>15000</td>
                <td>25</td>
            </tr>
        </tbody>
    </table>
</body>
</html>
"""

with open('temp_table.html', 'w', encoding='utf-8') as f:
    f.write(html_content)   

In [43]:
import os

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
file_path = "file:///" + os.path.abspath('temp_table.html')
driver.get(file_path)

# Находим таблицу
table = driver.find_element(By.ID, "sales-table")

# Извлекаем заголовки
headers = table.find_elements(By.TAG_NAME, "th")
header_names = [h.text for h in headers]
print("Заголовки:", header_names)

# Извлекаем все строки данных
rows = table.find_elements(By.CSS_SELECTOR, "tbody tr")
data = []

for row in rows:
    cells = row.find_elements(By.TAG_NAME, "td")
    row_data = [cell.text for cell in cells]
    data.append(row_data)
    print(row_data)

driver.quit()

# Создаём DataFrame
df = pd.DataFrame(data, columns=header_names)
print("\nDataFrame:")
print(df)

Заголовки: ['Продукт', 'Цена', 'Количество']
['Ноутбук', '50000', '15']
['Мышь', '500', '120']
['Клавиатура', '1500', '80']
['Монитор', '15000', '25']

DataFrame:
      Продукт   Цена Количество
0     Ноутбук  50000         15
1        Мышь    500        120
2  Клавиатура   1500         80
3     Монитор  15000         25


In [44]:
import os

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
driver.get("https://www.scrapethissite.com/pages/simple/")

countries = driver.find_elements(By.CLASS_NAME, "country")

data = []

for country in countries:
    name = country.find_element(By.CLASS_NAME, "country-name").text
    capital = country.find_element(By.CLASS_NAME, "country-capital").text
    population = country.find_element(By.CLASS_NAME, "country-population").text
    area = country.find_element(By.CLASS_NAME, "country-area").text
    
    data.append({
        "Country": name,
        "Capital": capital,
        "Population": population,
        "Area (km2)": area
    })
    
driver.quit()

df = pd.DataFrame(data)
print(df.head())

                Country           Capital Population Area (km2)
0               Andorra  Andorra la Vella      84000      468.0
1  United Arab Emirates         Abu Dhabi    4975593    82880.0
2           Afghanistan             Kabul   29121286   647500.0
3   Antigua and Barbuda        St. John's      86754      443.0
4              Anguilla        The Valley      13254      102.0


## Работа с динамическим контентом

In [46]:
dynamic_html = """
<!DOCTYPE html>
<html>
<head><title>Динамический контент</title></head>
<body>
    <h1>Список элементов</h1>
    <button id="load-btn" onclick="loadData()">Загрузить данные</button>
    <div id="content"></div>
    
    <script>
        function loadData() {
            setTimeout(() => {
                document.getElementById('content').innerHTML = 
                    '<p class="item">Элемент 1</p>' +
                    '<p class="item">Элемент 2</p>' +
                    '<p class="item">Элемент 3</p>';
            }, 2000);  // Загрузка через 2 секунды
        }
    </script>
</body>
</html>
"""

with open('temp_dynamic.html', 'w', encoding='utf-8') as f:
    f.write(dynamic_html)


In [47]:
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
file_path = "file:///" + os.path.abspath('temp_dynamic.html')
driver.get(file_path)

button = driver.find_element(By.ID, "load-btn")
button.click()

wait = WebDriverWait(driver, 10)
wait.until(EC.presence_of_element_located((By.CLASS_NAME, "item")))

items = driver.find_elements(By.CLASS_NAME, "item")
print(f"\nЗагружено элементов: {len(items)}")
for item in items:
    print(f"- {item.text}")

driver.quit()


Загружено элементов: 3
- Элемент 1
- Элемент 2
- Элемент 3


##  Прокрутка страницы и множественные действия

In [48]:
scroll_html = """
<!DOCTYPE html>
<html>
<head><title>Длинная страница</title></head>
<body>
    <h1>Начало страницы</h1>
""" + "\n".join([f"<p>Параграф {i}</p>" for i in range(1, 101)]) + """
    <div id="footer" style="background: yellow; padding: 20px;">
        <h2>Конец страницы</h2>
    </div>
</body>
</html>
"""

with open('temp_scroll.html', 'w', encoding='utf-8') as f:
    f.write(scroll_html)

In [ ]:
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
file_path = "file:///" + os.path.abspath('temp_scroll.html')
driver.get(file_path)

print("Страница открыта")
time.sleep(1)

print("Прокручиваем вниз...")   
driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
time.sleep(1)
    
footer = driver.find_element(By.ID, "footer")
print(f"Нашли текст: {footer.text}")

print("Прокручиваем наверх...")
driver.execute_script("window.scrollTo(0, 0);")
time.sleep(1)

driver.quit()

Страница открыта
Прокручиваем вниз...
Нашли текст: Конец страницы
Прокручиваем наверх...


In [ ]:
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
driver.get('https://www.scrapethissite.com/pages/ajax-javascript')
button = driver.find_element(By.ID, "2014")
button.click()


wait = WebDriverWait(driver,10)
wait.until(EC.presence_of_element_located((By.CLASS_NAME, "film")))

table = driver.find_element(By.CLASS_NAME, "table")
headers = driver.find_elements(By.CSS_SELECTOR, "thead th")
headers_names = [h.text for h in headers]
print(headers_names)

rows = table.find_elements(By.CSS_SELECTOR, "tbody tr")
data = []
for row in rows:
    cells = row.find_elements(By.TAG_NAME, "td")
    data_element = [cell.text for cell in cells]
    data.append(data_element)
    print(data)
    
driver.quit()
data_2014 = pd.DataFrame(data, columns = headers_names)

['Title', 'Nominations', 'Awards', 'Best Picture']
[['Birdman', '9', '4', '']]
[['Birdman', '9', '4', ''], ['The Grand Budapest Hotel', '9', '4', '']]
[['Birdman', '9', '4', ''], ['The Grand Budapest Hotel', '9', '4', ''], ['Whiplash', '5', '3', '']]
[['Birdman', '9', '4', ''], ['The Grand Budapest Hotel', '9', '4', ''], ['Whiplash', '5', '3', ''], ['The Imitation Game', '8', '1', '']]
[['Birdman', '9', '4', ''], ['The Grand Budapest Hotel', '9', '4', ''], ['Whiplash', '5', '3', ''], ['The Imitation Game', '8', '1', ''], ['American Sniper', '6', '1', '']]
[['Birdman', '9', '4', ''], ['The Grand Budapest Hotel', '9', '4', ''], ['Whiplash', '5', '3', ''], ['The Imitation Game', '8', '1', ''], ['American Sniper', '6', '1', ''], ['Boyhood', '6', '1', '']]
[['Birdman', '9', '4', ''], ['The Grand Budapest Hotel', '9', '4', ''], ['Whiplash', '5', '3', ''], ['The Imitation Game', '8', '1', ''], ['American Sniper', '6', '1', ''], ['Boyhood', '6', '1', ''], ['Interstellar', '5', '1', '']]
[['Bir

In [58]:
data_2014.head()

,Title,Nominations,Awards,Best Picture
0,Birdman,9,4,
1,The Grand Budapest Hotel,9,4,
2,Whiplash,5,3,
3,The Imitation Game,8,1,
4,American Sniper,6,1,


In [ ]:
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
driver.get('https://www.scrapethissite.com/pages/forms')

input_field = driver.find_element(By.CLASS_NAME, "form-control")
input_field.send_keys("Boston")
button = driver.find_element(By.CSS_SELECTOR, "input.btn.btn-primary")
button.click()
first_element = driver.find_element(By.CLASS_NAME, "team")
print(first_element.text)
driver.quit()

Boston Bruins 1990 44 24 0.55 299 264 35


# Задание. Решить задачу из https://www.scrapethissite.com/pages/